# 11: O Consolidador (relatório final)

Toda análise termina no Consolidador. Antes dele, a resposta final era só a última mensagem do grafo:
às vezes o texto do setor, às vezes o diagnóstico, nunca tudo junto.

O Consolidador **não é um agente com tools**: é uma única chamada ao LLM com saída estruturada
(`RelatorioFinal`), seguida de ajustes em código e da formatação em markdown.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. Uma análise completa da VALE3

Esta célula roda o pipeline inteiro (cerca de 30 segundos). A VALE3 costuma acionar o Investigator, então o
relatório terá diagnóstico.

In [2]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro import app

resultado = app.invoke({"messages": [HumanMessage("Analise a ação VALE3")]})
quem_falou = [m.name for m in resultado["messages"] if m.name]
print("Mensagens no histórico:", quem_falou)
print("loop_count:", resultado["loop_count"])

2026-11-07 23:31:05,725 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-07 23:31:06,836 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços, balanço patrimonial e principais métricas fundamentalistas para VALE3.'


2026-11-07 23:31:06,837 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-07 23:31:06,839 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


2026-11-07 23:31:07,575 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=VALE3.SA


2026-11-07 23:31:07,580 - fluxo_financeiro - INFO - [tool] calcular_metricas_risco: ticker=VALE3.SA


... (32 outputs left out)

2026-11-07 23:31:29,811 - fluxo_financeiro - DEBUG - [router] FINISH → Consolidador


2026-11-07 23:31:29,812 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (9 msgs)


2026-11-07 23:31:31,991 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (3374 chars)


Mensagens no histórico: ['Supervisor', 'YFinance', 'Supervisor', 'Analyzer', 'Supervisor', 'Analyzer', 'Supervisor', 'Investigator', 'Consolidador']
loop_count: 5


## 2. O que o Consolidador recebe e devolve

Ele recebe só o pedido do usuário e as respostas de YFinance, Analyzer e Investigator (as tarefas do
Supervisor ficam de fora), cada uma dentro de `<resposta_agente>`. Devolve um `RelatorioFinal`:

In [3]:
from fluxo_financeiro.agents.consolidator import montar_insumos

print("\n\n".join(montar_insumos(resultado["messages"]))[:1500], "...")

<pedido_usuario>Analise a ação VALE3</pedido_usuario>

<resposta_agente nome="YFinance" n="1">
Dados para VALE3.SA:
Preço Atual: R$ 64.04
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Industria: Other Industrial Metals & Mining
Sumario de Negócio:
Vale S.A., together with its subsidiaries, produces iron ore and nickel in Brazil, Asia, the Middle East, North Africa, Europe, the Americas, and Oceania. The company operates in two segments, Iron Ore Solutions and Vale Base Metals. It extracts, produces, and distributes iron ore, iron ore pellets, briquettes, nickel, copper, other ferrous products, and by-products, including gold, silver, cobalt, platinum-group metals, and other base metals, as well as low-carbon critical minerals. The company also operates logistics systems and distribution centers, such as mining complexes, railways, and maritime terminals, ports, and ships; generates energy from hydroelectric, solar, and wind sources; and engages in greenfield mineral exploration. In addition, it i

In [4]:
relatorio = resultado["relatorio"]
for campo in ("resumo", "indicadores", "contexto", "riscos", "limitacoes"):
    print(f"--- {campo}:\n{getattr(relatorio, campo)}\n")

--- resumo:
A ação VALE3 é negociada a R$ 64,04, refletindo uma recuperação recente após oscilações entre R$ 60,25 e o patamar atual. O desempenho da mineradora foi pressionado no trimestre por preços globais mais fracos do minério de ferro e maiores custos operacionais, resultando em uma queda de 53% no lucro líquido ano contra ano. Apesar disso, a companhia mantém uma geração de caixa operacional robusta e perspectivas favoráveis ligadas à transição energética.

--- indicadores:
['Preço Atual: R$ 64.04', 'P/L: 77.2x', 'ROE: 0.3%', 'DY: 6.82%', 'Sharpe Ratio: -0.11', 'Sortino Ratio: -0.18', 'Max Drawdown: -31.5%', 'VaR 95% (diário): -2.91%', 'Beta (vs IBOV): 0.39', 'Alpha (anual): -3.4%', 'Piotroski F-Score: 5/9', 'Net Income: US$ 2.35B', 'Operating Cash Flow: US$ 8.80B', 'Total Assets cresceu: +8.0%', 'Divida corrente aumentou: +21.2%', 'Current Ratio: 1.15', 'Gross Margin: 35.0%']

--- contexto:
A Vale atua nos segmentos de soluções de ferro e metais básicos, produzindo minério de f

## 3. O diagnóstico é copiado, não resumido

O LLM é instruído a deixar `diagnostico` vazio; o código copia o texto do Investigator literalmente. Assim as
quatro seções (Causa, Natureza, Evidências, Impacto) nunca são reescritas.

In [5]:
texto_investigator = next((m.text for m in resultado["messages"] if m.name == "Investigator"), None)
print("Investigator rodou:", texto_investigator is not None)
print("Diagnóstico idêntico ao texto do Investigator:", relatorio.diagnostico == texto_investigator)

Investigator rodou: True
Diagnóstico idêntico ao texto do Investigator: True


## 4. A mesma função, chamada à parte

`consolidar(messages)` é uma função comum: dá para chamá-la com qualquer histórico. Uma chamada ao LLM.

In [6]:
from fluxo_financeiro.agents.consolidator import consolidar, formatar_relatorio, relatorio_de_emergencia

de_novo = consolidar(resultado["messages"])
print("Indicadores na 2a chamada:", len(de_novo.indicadores), "| na análise:", len(relatorio.indicadores))

Indicadores na 2a chamada: 11 | na análise: 17


## 5. Do relatório estruturado ao markdown

`formatar_relatorio` monta o markdown em código. Seções vazias são omitidas.

In [7]:
from IPython.display import Markdown

Markdown(formatar_relatorio(relatorio, "VALE3"))

# Relatório: VALE3

## Resumo
A ação VALE3 é negociada a R$ 64,04, refletindo uma recuperação recente após oscilações entre R$ 60,25 e o patamar atual. O desempenho da mineradora foi pressionado no trimestre por preços globais mais fracos do minério de ferro e maiores custos operacionais, resultando em uma queda de 53% no lucro líquido ano contra ano. Apesar disso, a companhia mantém uma geração de caixa operacional robusta e perspectivas favoráveis ligadas à transição energética.

## Indicadores
- Preço Atual: R$ 64.04
- P/L: 77.2x
- ROE: 0.3%
- DY: 6.82%
- Sharpe Ratio: -0.11
- Sortino Ratio: -0.18
- Max Drawdown: -31.5%
- VaR 95% (diário): -2.91%
- Beta (vs IBOV): 0.39
- Alpha (anual): -3.4%
- Piotroski F-Score: 5/9
- Net Income: US$ 2.35B
- Operating Cash Flow: US$ 8.80B
- Total Assets cresceu: +8.0%
- Divida corrente aumentou: +21.2%
- Current Ratio: 1.15
- Gross Margin: 35.0%

## Notícias e setor
A Vale atua nos segmentos de soluções de ferro e metais básicos, produzindo minério de ferro, níquel, cobre e minerais críticos para a transição energética, além de operar sistemas logísticos e portuários. No setor de metais e mineração industrial, a companhia enfrenta desafios comuns como a elevação de custos logísticos e de frete. Fatores recentes que moldam o contexto incluem a retração nos preços internacionais do minério de ferro devido à demanda fraca da China, a apreciação do Real frente ao dólar e a aprovação da Política Nacional de Minerais Críticos e Estratégicos (PNMCE) pelo governo federal, que prioriza substâncias como cobre e níquel.

## Diagnóstico da anomalia
- Causa identificada: A queda de 53% no lucro líquido da VALE3 no 3T26 (US$ 1,25 bi versus US$ 2,69 bi no 3T25) foi motivada principalmente por fatores macroeconômicos e operacionais desfavoráveis, destacando-se a retração nos preços internacionais do minério de ferro (pressionados pela demanda fraca da China), o aumento nos custos operacionais logísticos e de extração, a apreciação do Real (BRL) frente ao dólar e uma leve redução na produção por menor disponibilidade de minério em Serra Norte.
- Natureza do evento: **Cíclico e Operacional** (associado diretamente à dinâmica de preços de commodities, pressões inflacionárias de custos e flutuações cambiais, embora com pressões recorrentes de margem).
- Evidências: Fatos relevantes e relatórios de resultados divulgados pela companhia, dados obtidos via buscas de fatos corporativos e histórico trimestral de demonstrações financeiras (yfinance).
- Impacto no investimento: O resultado evidencia a vulnerabilidade da mineradora a ciclos de baixa de preços do minério de ferro e inflação de custos logísticos, limitando o espaço para distribuição de dividendos extraordinários no curto prazo e elevando os múltiplos de curto prazo (como o P/L), embora a geração operacional de caixa permaneça resiliente (US$ 8,80B no acumulado) e o balanço mantenha boa saúde financeira (F-Score 5/9).

## Riscos
- Retração nos preços internacionais do minério de ferro e pressões da demanda fraca da China
- Aumento nos custos operacionais logísticos, de extração e de frete
- Múltiplo P/L elevado de 77,2x (quantos anos de lucro pagam o preço da ação)
- Max drawdown de -31.5% e índice Sharpe de -0.11 no último ano
- Aumento de 21.2% na dívida corrente
- Vulnerabilidade a flutuações cambiais (apreciação do Real frente ao dólar)


## 6. Se o LLM do Consolidador falhar

A análise já custou várias chamadas; perder tudo no último passo seria ruim. Se `consolidar` levantar erro
(cota, rede, saída inválida), o nó usa `relatorio_de_emergencia`: as respostas dos agentes juntadas, sem LLM,
e `relatorio = None` no estado.

In [8]:
print(relatorio_de_emergencia(resultado["messages"])[:800], "...")

> Não foi possível consolidar o relatório; seguem as respostas dos agentes.

## YFinance
Dados para VALE3.SA:
Preço Atual: R$ 64.04
P/L:  77.2x
ROE:  0.3%
DY:   6.82%
Industria: Other Industrial Metals & Mining
Sumario de Negócio:
Vale S.A., together with its subsidiaries, produces iron ore and nickel in Brazil, Asia, the Middle East, North Africa, Europe, the Americas, and Oceania. The company operates in two segments, Iron Ore Solutions and Vale Base Metals. It extracts, produces, and distributes iron ore, iron ore pellets, briquettes, nickel, copper, other ferrous products, and by-products, including gold, silver, cobalt, platinum-group metals, and other base metals, as well as low-carbon critical minerals. The company also operates logistics systems and distribution centers, such as mi ...


## Sem recomendação de compra ou venda

O relatório descreve o que os dados mostram e não dá veredito nem preço-alvo. Um "compre" escrito por um LLM a
partir de poucas fontes passaria uma confiança que os dados não sustentam.

## O que observar

- `limitacoes`: lista o que faltou ou falhou (busca sem resultado, etapa que não rodou). Parte delas é escrita em código, não depende de o LLM perceber.
- Os números de `indicadores` precisam aparecer nas respostas dos agentes: o prompt proíbe inventar ou recalcular.
- A segunda chamada a `consolidar` pode escrever frases diferentes da primeira (o LLM não é determinístico), mas os números são os mesmos, porque vêm dos insumos.
- O relatório de emergência é feio, mas não perde nada do que os agentes produziram.